In [35]:
from dotenv import load_dotenv
import os


load_dotenv()



OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
OPEN_ROUTER_COMPLETION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"


In [36]:
import openai
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPEN_ROUTER_API_KEY)

In [37]:
import json

In [38]:
SYSTEM_META_PROMPT="""
You are a professional children's story writer.

Your goals:
- Write bedtime stories for children aged 4–9.
- Stories should be calming.
- Never include violence or horror.
- Keep language simple.
- If you need to use a tool, use it.
- Stories should be relaxing, pleasing to hear and lesson full.
"""

In [45]:
META_PROMPT_OPTIMIZER="""
You are a professional children's story writer.

Your goals:
- Write bedtime stories for children aged 4–9.
- Return an optimized prompt for generating stories
"""
async def PromptOptimizerTool(prompt: str):
        chat_completion = client.chat.completions.create(
            model=OPEN_ROUTER_COMPLETION_MODEL,
            messages=[
                {"role": "system", "content": META_PROMPT_OPTIMIZER},
                {
                    "role": "user",
                    "content": f"{prompt}",
                },
            ],
            tools=tools,
        )

        response_message = chat_completion.choices[0].message
        
        print("PromptOptimizerTool called")
        print(response_message)

In [40]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "PromptOptimizerTool",
            "description": "Optimize user input prompt to a level it creates stunning storyline",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "User input prompt to optimize",
                    }
                },
                "required": ["prompt"],
            },
        },
    }
]

In [46]:
async def openai_chat_completion(prompt: str, META_PROMPT:str):
    try:
        chat_completion = client.chat.completions.create(
            model=OPEN_ROUTER_COMPLETION_MODEL,
            messages=[
                {"role": "system", "content": META_PROMPT},
                {
                    "role": "user",
                    "content": f"{prompt}",
                },
            ],
            tools=tools,
        )

        response_message = chat_completion.choices[0].message
        # message_content = response_message.content or ""
        # self.messages.append(response_message.model_dump())

        # If LLM returned tool calls, process them
        if hasattr(response_message, 'tool_calls') and response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                
                print(f"Tool call: {function_name}, args: {function_args}")
                agent_args = []
                
                tool_function = globals()[function_name]
                tool_result = await tool_function(*agent_args, **function_args)
                print(tool_result)
        
        return chat_completion

    except openai.APIConnectionError as e:
        print(f"Network connectivity issue: {e}")
    except openai.RateLimitError as e:
        print(f"Rate limits hit or out of funds: {e}")
    except openai.APIStatusError as e:
        print(f"HTTP Error received (Status: {e.status_code}): {e.response}")

In [47]:
await openai_chat_completion('rabbit', SYSTEM_META_PROMPT)

Tool call: PromptOptimizerTool, args: {'prompt': 'rabbit'}
PromptOptimizerTool called
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-b9567bda-a3f1-4462-af67-396550318741', function=Function(arguments='{"prompt":"rabbit"}', name='PromptOptimizerTool'), type='function', index=0)], reasoning='The user just said "rabbit". This is a very minimal prompt. As a children\'s story writer, I should optimize this into a proper story prompt. I\'ll use the PromptOptimizerTool to expand "rabbit" into a full story concept suitable for ages 4-9', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user just said "rabbit". This is a very minimal prompt. As a children\'s story writer, I should optimize this into a proper story prompt. I\'ll use the PromptOptimizerTool to expand "rabbit" into a full story concept suitable for ages 4-9', 'format': 'unknown', 'index': 0}])

ChatCompletion(id='gen-1784744444-OTQXYoBZMbQXFt4t0czI', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-46e86f80-1f43-49b2-a04f-aa9741ea63b7', function=Function(arguments='{"prompt":"rabbit"}', name='PromptOptimizerTool'), type='function', index=0)], reasoning='The user just said "rabbit" - they want a bedtime story about a rabbit. I should use the PromptOptimizerTool to optimize this simple prompt into a more detailed storyline for a children\'s bedtime story.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user just said "rabbit" - they want a bedtime story about a rabbit. I should use the PromptOptimizerTool to optimize this simple prompt into a more detailed storyline for a children\'s bedtime story.', 'format': 'unknown', 'index': 0}]), native_finish_reason='tool_ca

In [43]:
# ChatCompletion(
#     id="gen-1784740420-a3TdQhM44Qqj99iyUHf7",
#     # choices=[
#     #     Choice(
#     #         finish_reason="tool_calls",
#     #         index=0,
#     #         logprobs=None,
#     #         message=ChatCompletionMessage(
#     #             content=None,
#     #             refusal=None,
#     #             role="assistant",
#     #             annotations=None,
#     #             audio=None,
#     #             function_call=None,
#     #             tool_calls=[
#     #                 ChatCompletionMessageFunctionToolCall(
#     #                     id="call-251336db-fc7d-4afb-a457-63afc788d290",
#     #                     function=Function(
#     #                         arguments='{"prompt":"rabbit"}', name="PromptOptimizerTool"
#     #                     ),
#     #                     type="function",
#     #                     index=0,
#     #                 )
#     #             ],
#     #             reasoning="The user wants a bedtime story about a rabbit. I should use the PromptOptimizerTool to optimize this simple prompt into a better storyline before writing the story. Let me do that first.",
#     #             reasoning_details=[
#     #                 {
#     #                     "type": "reasoning.text",
#     #                     "text": "The user wants a bedtime story about a rabbit. I should use the PromptOptimizerTool to optimize this simple prompt into a better storyline before writing the story. Let me do that first.",
#     #                     "format": "unknown",
#     #                     "index": 0,
#     #                 }
#     #             ],
#     #         ),
#     #         native_finish_reason="tool_calls",
#     #     )
#     # ],
#     created=1784740420,
#     model="nvidia/nemotron-3-ultra-550b-a55b:free",
#     object="chat.completion",
#     moderation=None,
#     service_tier=None,
#     system_fingerprint=None,
#     usage=CompletionUsage(
#         completion_tokens=68,
#         prompt_tokens=347,
#         total_tokens=415,
#         completion_tokens_details=CompletionTokensDetails(
#             accepted_prediction_tokens=None,
#             audio_tokens=0,
#             reasoning_tokens=47,
#             rejected_prediction_tokens=None,
#             image_tokens=0,
#         ),
#         prompt_tokens_details=PromptTokensDetails(
#             audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0
#         ),
#         cost=0,
#         is_byok=False,
#         cost_details={
#             "upstream_inference_cost": 0,
#             "upstream_inference_prompt_cost": 0,
#             "upstream_inference_completions_cost": 0,
#         },
#     ),
#     provider="Nvidia",
# )

In [44]:
await openai_chat_completion('hi')

TypeError: openai_chat_completion() missing 1 required positional argument: 'META_PROMPT'